# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Moataz-Elzuhery/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*


**Unit of analysis:** One row represents one content item associated with one anonymized client.

**Time window:** This dataset is a content-level snapshot. It does not contain a row-level calendar date, so an exact calendar date range cannot be stated. Performance fields use fixed windows such as 90 days, with some fields comparing the last 30 days with the previous 30 days.




In [32]:
# Check the dataset grain and available time-window fields

print("Rows:", con.execute("""
    SELECT COUNT(*)
    FROM content_refresh
""").fetchone()[0])

print("Unique content IDs:", con.execute("""
    SELECT COUNT(DISTINCT content_id)
    FROM content_refresh
""").fetchone()[0])

print("Unique client IDs:", con.execute("""
    SELECT COUNT(DISTINCT client_id)
    FROM content_refresh
""").fetchone()[0])

Rows: 30000
Unique content IDs: 30000
Unique client IDs: 32


In [33]:
import os
import duckdb
import pandas as pd
from google.colab import userdata

# Get Hugging Face token
hf_token = userdata.get("HF_TOKEN")

assert hf_token, "HF_TOKEN is missing from Colab Secrets"

# Connect to DuckDB
con = duckdb.connect()

# Install and load extensions
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")



# Create Hugging Face authentication secret
con.execute(f"""
CREATE SECRET hf_auth (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
)
""")

print("DuckDB + Hugging Face connection is ready.")



DuckDB + Hugging Face connection is ready.


In [34]:
# Check DuckDB version and available extensions
print("DuckDB version:", duckdb.__version__)

print(
    con.execute("SELECT * FROM duckdb_extensions()")
    .df()[["extension_name", "loaded", "installed"]]
)

DuckDB version: 1.5.5
      extension_name  loaded  installed
0       autocomplete   False      False
1               avro   False      False
2                aws   False      False
3              azure   False      False
4     core_functions    True       True
5              delta   False      False
6           ducklake   False      False
7          encodings   False      False
8              excel   False      False
9                fts   False      False
10            httpfs    True       True
11           iceberg   False      False
12               icu    True       True
13              inet   False      False
14              json    True       True
15             lance   False      False
16        motherduck   False      False
17     mysql_scanner   False      False
18      odbc_scanner   False      False
19           parquet    True       True
20  postgres_scanner   False      False
21             quack   False      False
22           spatial   False      False
23    sqlite_scann

In [35]:
# Test HTTP access
print(con.execute("""
    SELECT *
    FROM read_csv_auto(
        'https://raw.githubusercontent.com/Moataz-Elzuhery/flyrank-ml-internship/main/README.md'
    )
    LIMIT 5
""").df())

              # FlyRank ML Internship — Starter Repo
0                                               None
1  **Applied Search Intelligence: Google Search R...
2                                               None
3  This is the starting point for the FlyRank ML ...
4  repo** (one click — *Use this template*), buil...


In [36]:
# List files in the starter repo
import requests

repo_url = "https://api.github.com/repos/Moataz-Elzuhery/flyrank-ml-internship/git/trees/main?recursive=1"

files = requests.get(repo_url).json()["tree"]

for f in files:
    print(f["path"])

.github
.github/workflows
.github/workflows/data-path-smoke.yml
.github/workflows/personalize.yml
.github/workflows/smoke-test.yml
.gitignore
AGENTS.md
CLAUDE.md
DATA_USE.md
GUIDE.md
LICENSE
README.md
SETUP.md
data
data/raw
data/raw/content_refresh_anonymized.csv
docs
docs/data-dictionary.md
docs/flyrank-seo-research-march-2026.pdf
docs/intern-free-tooling-guide.md
docs/ml-core-foundation-framework.md
docs/ml-intern-dataset-and-lane-guide.md
docs/paper-examples
docs/paper-examples/README.md
docs/paper-examples/index.html
notebooks
notebooks/01_first_look_and_discovery.ipynb
notebooks/02_your_first_readable_model.ipynb
notebooks/03_working_with_the_full_release.ipynb
outputs
outputs/charts
outputs/charts/action_mix.svg
outputs/charts/confidence_mix.svg
outputs/charts/top_feature_importance.svg
outputs/charts/top_reason_codes.svg
outputs/charts/trend_distribution.svg
outputs/model_report.md
outputs/refresh_queue_sample.csv
requirements.txt
scripts
scripts/01_prepare_features.py
scripts/0

In [37]:
DATA_PATH = "https://raw.githubusercontent.com/Moataz-Elzuhery/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = con.execute(f"""
    SELECT *
    FROM read_csv_auto('{DATA_PATH}')
    LIMIT 5
""").df()

df

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10,0.67,HIGH,2.05,keyword article,transactional,3221,20457,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90,0.01,LOW,0.05,keyword article,informational,2481,15562,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0,0.00,LOW,0.00,keyword article,informational,3515,23643,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10,0.00,LOW,0.00,keyword article,commercial,<NA>,<NA>,...,None,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0,0.00,LOW,0.00,keyword article,informational,2803,17469,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [38]:
con.execute(f"""
    CREATE OR REPLACE VIEW content_refresh AS
    SELECT *
    FROM read_csv_auto('{DATA_PATH}')
""")

print(con.execute("DESCRIBE content_refresh").df())

               column_name column_type null   key default extra
0               content_id     VARCHAR  YES  None    None  None
1                client_id     VARCHAR  YES  None    None  None
2            search_volume      BIGINT  YES  None    None  None
3              competition      DOUBLE  YES  None    None  None
4        competition_level     VARCHAR  YES  None    None  None
5                      cpc      DOUBLE  YES  None    None  None
6             content_type     VARCHAR  YES  None    None  None
7              main_intent     VARCHAR  YES  None    None  None
8               word_count      BIGINT  YES  None    None  None
9               char_count      BIGINT  YES  None    None  None
10           provider_used     VARCHAR  YES  None    None  None
11              model_used     VARCHAR  YES  None    None  None
12         impressions_90d      BIGINT  YES  None    None  None
13              clicks_90d      BIGINT  YES  None    None  None
14           pageviews_90d      BIGINT  

In [39]:
con.execute("""
    SELECT COUNT(*) AS rows
    FROM content_refresh
""").df()

,rows
0,30000


In [40]:
con.execute("""
    SELECT *
    FROM content_refresh
    LIMIT 10
""").df()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10,0.67,HIGH,2.05,keyword article,transactional,3221,20457,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90,0.01,LOW,0.05,keyword article,informational,2481,15562,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0,0.00,LOW,0.00,keyword article,informational,3515,23643,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10,0.00,LOW,0.00,keyword article,commercial,<NA>,<NA>,...,None,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0,0.00,LOW,0.00,keyword article,informational,2803,17469,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,720,1.00,HIGH,1.05,keyword article,transactional,3080,18178,...,15000-25000,0.03,8.5,0.00,25.00,0.0,good,page_1,down,-38.9
6,content_9a34b442b552,client_8722616204,0,0.00,LOW,0.00,keyword article,informational,3059,20810,...,15000-25000,0.00,7.0,0.00,0.00,0.0,low,page_1,down,-92.3
7,content_a63219c6e95a,client_19581e27de,590,0.44,MEDIUM,0.64,keyword article,commercial,<NA>,<NA>,...,None,0.06,21.2,3.57,7.14,0.0,moderate,page_3_5,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,0,0.00,LOW,0.00,keyword article,informational,3807,24228,...,15000-25000,0.09,46.0,5.88,6.25,0.0,excellent,page_3_5,down,-58.8
9,content_c27558df2b0c,client_19581e27de,0,0.00,LOW,0.00,keyword article,informational,<NA>,<NA>,...,None,0.16,4.9,0.00,0.00,0.0,moderate,page_1,down,-29.2


In [41]:
con.execute("DESCRIBE content_refresh").df()

,column_name,column_type,null,key,default,extra
0,content_id,VARCHAR,YES,None,None,None
1,client_id,VARCHAR,YES,None,None,None
2,search_volume,BIGINT,YES,None,None,None
3,competition,DOUBLE,YES,None,None,None
4,competition_level,VARCHAR,YES,None,None,None
5,cpc,DOUBLE,YES,None,None,None
6,content_type,VARCHAR,YES,None,None,None
7,main_intent,VARCHAR,YES,None,None,None
8,word_count,BIGINT,YES,None,None,None
9,char_count,BIGINT,YES,None,None,None


In [42]:
con.execute("""
    SELECT *
    FROM content_refresh
    LIMIT 10
""").df()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10,0.67,HIGH,2.05,keyword article,transactional,3221,20457,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90,0.01,LOW,0.05,keyword article,informational,2481,15562,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0,0.00,LOW,0.00,keyword article,informational,3515,23643,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10,0.00,LOW,0.00,keyword article,commercial,<NA>,<NA>,...,None,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0,0.00,LOW,0.00,keyword article,informational,2803,17469,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7
5,content_d4084a4bc775,client_f369cb89fc,720,1.00,HIGH,1.05,keyword article,transactional,3080,18178,...,15000-25000,0.03,8.5,0.00,25.00,0.0,good,page_1,down,-38.9
6,content_9a34b442b552,client_8722616204,0,0.00,LOW,0.00,keyword article,informational,3059,20810,...,15000-25000,0.00,7.0,0.00,0.00,0.0,low,page_1,down,-92.3
7,content_a63219c6e95a,client_19581e27de,590,0.44,MEDIUM,0.64,keyword article,commercial,<NA>,<NA>,...,None,0.06,21.2,3.57,7.14,0.0,moderate,page_3_5,stable,0.6
8,content_5e6c160719bc,client_6208ef0f77,0,0.00,LOW,0.00,keyword article,informational,3807,24228,...,15000-25000,0.09,46.0,5.88,6.25,0.0,excellent,page_3_5,down,-58.8
9,content_c27558df2b0c,client_19581e27de,0,0.00,LOW,0.00,keyword article,informational,<NA>,<NA>,...,None,0.16,4.9,0.00,0.00,0.0,moderate,page_1,down,-29.2


In [43]:
columns = con.execute("DESCRIBE content_refresh").df()
columns[["column_name", "column_type"]]

,column_name,column_type
0,content_id,VARCHAR
1,client_id,VARCHAR
2,search_volume,BIGINT
3,competition,DOUBLE
4,competition_level,VARCHAR
5,cpc,DOUBLE
6,content_type,VARCHAR
7,main_intent,VARCHAR
8,word_count,BIGINT
9,char_count,BIGINT


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

I classify the fields into four groups:

* **Features:** fields that can be used as inputs for the ML task.
* **Label:** the outcome we want to predict or use as the target.
* **Context:** fields useful for interpreting the data but not used as model inputs.
* **Excluded:** fields I will not use because they may cause leakage, identify private information, or are not relevant to the task.

**Features:**
`search_volume`, `competition`, `competition_level`, `cpc`, `content_type`, `main_intent`, `word_count`, `char_count`, `content_age_days`, `age_tier`, `days_since_last_update`, `freshness_tier`, `word_count_tier`, `char_count_tier`

**Label:**
`trend_direction`

**Context:**
`content_id`, `client_id`, `impressions_90d`, `clicks_90d`, `pageviews_90d`, `sessions_90d`, `users_90d`, `engaged_sessions_90d`, `ai_sessions_90d`, `scroll_events_90d`, `days_with_impressions`, `days_with_sessions`, `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `impression_tier`, `position_tier`, `trend_pct`

**Excluded:**
`provider_used`, `model_used`

These are excluded because they describe the provider/model used to generate or process the content rather than the content's search characteristics. They could also make the analysis depend on the generation setup rather than the search problem.


In [44]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [45]:
# Basic data contract checks

print("Rows:")
print(con.execute("""
    SELECT COUNT(*) AS rows
    FROM content_refresh
""").df())

print("\nUnique content/client combinations:")
print(con.execute("""
    SELECT
        COUNT(*) AS rows,
        COUNT(DISTINCT content_id) AS unique_content,
        COUNT(DISTINCT client_id) AS unique_clients,
        COUNT(DISTINCT (content_id, client_id)) AS unique_content_client
    FROM content_refresh
""").df())

print("\nTrend direction distribution:")
print(con.execute("""
    SELECT
        trend_direction,
        COUNT(*) AS rows
    FROM content_refresh
    GROUP BY trend_direction
    ORDER BY rows DESC
""").df())

print("\nMissing values in selected fields:")
print(con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) - COUNT(word_count) AS missing_word_count,
        COUNT(*) - COUNT(char_count) AS missing_char_count,
        COUNT(*) - COUNT(content_age_days) AS missing_content_age,
        COUNT(*) - COUNT(days_since_last_update) AS missing_last_update,
        COUNT(*) - COUNT(trend_direction) AS missing_trend_direction
    FROM content_refresh
""").df())


Rows:
    rows
0  30000

Unique content/client combinations:
    rows  unique_content  unique_clients  unique_content_client
0  30000           30000              32                  30000

Trend direction distribution:
  trend_direction   rows
0            down  16262
1          stable   5962
2              up   4388
3             new   2236
4            flat   1152

Missing values in selected fields:
   total_rows  missing_word_count  missing_char_count  missing_content_age  \
0       30000                7699                7699                    0   

   missing_last_update  missing_trend_direction  
0                    0                        0  


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset is a content-level snapshot rather than a full time-series dataset. It does not contain a row-level calendar date, so it cannot support claims about exact historical periods.

The performance fields are aggregated over fixed windows such as 90 days and compare recent 30-day activity with the previous 30-day period. These windows provide directional signals but do not establish causality.

The dataset also contains anonymized client and content IDs, so it cannot be used to identify specific clients or explain client-specific business context.

Results should therefore be treated as observed, measured, and directional decision-support rather than proof of causality or a complete representation of search behavior.



In [46]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.